# Assignment 4 Solution (Lab Session 5) Decision Tree

## Import the Modules

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from pprint import pprint


## Import the Bike Sharing Dataset (https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)

In [ ]:
!pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
bike_sharing = fetch_ucirepo(id=275)

# data (as pandas dataframes)
X = bike_sharing.data.features
y = bike_sharing.data.targets

# metadata
pprint(bike_sharing.metadata)

# variable information
print(bike_sharing.variables)


{'abstract': 'This dataset contains the hourly and daily count of rental bikes '
             'between years 2011 and 2012 in Capital bikeshare system with the '
             'corresponding weather and seasonal information.',
 'additional_info': {'citation': None,
                     'funded_by': None,
                     'instances_represent': None,
                     'preprocessing_description': None,
                     'purpose': None,
                     'recommended_data_splits': None,
                     'sensitive_data': None,
                     'summary': 'Bike sharing systems are new generation of '
                                'traditional bike rentals where whole process '
                                'from membership, rental and return back has '
                                'become automatic. Through these systems, user '
                                'is able to easily rent a bike from a '
                                'particular position and retur

## Process the Features(if needed)

In [ ]:
y = y["cnt"]  # pick total count as target
print(X)
print("Features:", X.columns.tolist())
print("Target:", y.name)

X = X.drop(columns=["dteday"])

cat_cols = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]
num_cols = ["temp", "atemp", "hum", "windspeed"]

enc = OneHotEncoder(handle_unknown="ignore")
X_cat = enc.fit_transform(X[cat_cols])
X_num = X[num_cols].to_numpy()

X_proc = np.hstack([X_num, X_cat.toarray()])


           dteday  season  yr  mnth  hr  holiday  weekday  workingday  \
0      2011-01-01       1   0     1   0        0        6           0   
1      2011-01-01       1   0     1   1        0        6           0   
2      2011-01-01       1   0     1   2        0        6           0   
3      2011-01-01       1   0     1   3        0        6           0   
4      2011-01-01       1   0     1   4        0        6           0   
...           ...     ...  ..   ...  ..      ...      ...         ...   
17374  2012-12-31       1   1    12  19        0        1           1   
17375  2012-12-31       1   1    12  20        0        1           1   
17376  2012-12-31       1   1    12  21        0        1           1   
17377  2012-12-31       1   1    12  22        0        1           1   
17378  2012-12-31       1   1    12  23        0        1           1   

       weathersit  temp   atemp   hum  windspeed  
0               1  0.24  0.2879  0.81     0.0000  
1               1  0.

## Split the dataset into training, validation and test sets (70%-15%-15%).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X_proc, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)


## Implement a decision tree regressor from scratch using reduction in MSE of the target (Count of total no of bikes) as the node splitting criterion.

In [ ]:
# --- Decision Tree Regressor from Scratch ---
class DecisionTreeRegressorScratch:
    def __init__(self, max_depth=5, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.tree = None

    def mse(self, y):
        return np.mean((y - np.mean(y))**2) if len(y) > 0 else 0

    def best_split(self, X, y):
        best_feature, best_threshold, best_mse, best_split = None, None, float("inf"), None
        n_samples, n_features = X.shape

        if n_samples < self.min_samples_split:
            return None, None, None, None

        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])
            for t in thresholds:
                left_mask = X[:, feature] <= t
                right_mask = X[:, feature] > t

                if (sum(left_mask) < self.min_samples_leaf or
                    sum(right_mask) < self.min_samples_leaf):
                    continue

                left_y, right_y = y[left_mask], y[right_mask]
                mse_split = (len(left_y) * self.mse(left_y) + len(right_y) * self.mse(right_y)) / n_samples

                if mse_split < best_mse:
                    best_feature, best_threshold, best_mse = feature, t, mse_split
                    best_split = (left_mask, right_mask)

        return best_feature, best_threshold, best_mse, best_split

    def build_tree(self, X, y, depth=0):
        if (depth >= self.max_depth or len(y) < self.min_samples_split or len(set(y)) == 1):
            return np.mean(y)

        feature, threshold, mse, split = self.best_split(X, y)
        if feature is None:
            return np.mean(y)

        left_mask, right_mask = split
        left_child = self.build_tree(X[left_mask], y[left_mask], depth+1)
        right_child = self.build_tree(X[right_mask], y[right_mask], depth+1)

        return {"feature": feature, "threshold": threshold, "left": left_child, "right": right_child}

    def fit(self, X, y):
        self.tree = self.build_tree(np.array(X), np.array(y))

    def predict_one(self, x, node):
        if not isinstance(node, dict):
            return node
        if x[node["feature"]] <= node["threshold"]:
            return self.predict_one(x, node["left"])
        else:
            return self.predict_one(x, node["right"])

    def predict(self, X):
        return np.array([self.predict_one(x, self.tree) for x in np.array(X)])

## Tune the hyperparameters of the decision tree (maximum depth of tree, minimum samples in node for split, minimum samples for leaf node) on the validation set using grid search

In [ ]:
param_grid = {
    "max_depth": [x for x in range(3,15)],
    "min_samples_split": [x for x in range(5,20,2)],
    "min_samples_leaf": [x for x in range(2,10)],
}

best_params = None
best_val_mse = float("inf")

for depth in param_grid["max_depth"]:
    for split in param_grid["min_samples_split"]:
        for leaf in param_grid["min_samples_leaf"]:
            model = DecisionTreeRegressorScratch(max_depth=depth,
                                                 min_samples_split=split,
                                                 min_samples_leaf=leaf)
            model.fit(X_train, y_train)
            preds = model.predict(X_val)
            mse_val = mean_squared_error(y_val, preds)
            if mse_val < best_val_mse:
                best_val_mse = mse_val
                best_params = (depth, split, leaf)
            print(f"Model done for, {depth} depth, {split} minimum sample split, {leaf} minimum sample leaf")
            print(f"MSE: {mse_val}")

print("Best scratch params:", best_params, "Validation MSE:", best_val_mse)

Model done for, 3 depth, 5 minimum sample split, 2 minimum sample leaf
MSE: 20391.531705097797
Model done for, 3 depth, 5 minimum sample split, 3 minimum sample leaf
MSE: 20391.531705097797
Model done for, 3 depth, 5 minimum sample split, 4 minimum sample leaf
MSE: 20391.531705097797
Model done for, 3 depth, 5 minimum sample split, 5 minimum sample leaf
MSE: 20391.531705097797
Model done for, 3 depth, 5 minimum sample split, 6 minimum sample leaf
MSE: 20391.531705097797
Model done for, 3 depth, 5 minimum sample split, 7 minimum sample leaf
MSE: 20391.531705097797
Model done for, 3 depth, 5 minimum sample split, 8 minimum sample leaf
MSE: 20391.531705097797
Model done for, 3 depth, 5 minimum sample split, 9 minimum sample leaf
MSE: 20391.531705097797
Model done for, 3 depth, 7 minimum sample split, 2 minimum sample leaf
MSE: 20391.531705097797
Model done for, 3 depth, 7 minimum sample split, 3 minimum sample leaf
MSE: 20391.531705097797
Model done for, 3 depth, 7 minimum sample split, 4

## Evaluate the performance of the decision tree regressor on the test set using the best values of the hyperparameters you obtained in step

In [ ]:
tree=DecisionTreeRegressorScratch(*best_params)
tree.fit(X_train, y_train)
y_pred = tree.predict(X_test)

mse_test_scratch=mean_squared_error(y_test, y_pred)
print("MSE:", mse_test_scratch)
print("R2 Score:", r2_score(y_test, y_pred))




NameError: name 'DecisionTreeRegressorScratch' is not defined

## Report the MSE obtained on the test set and show the scatter plot of predictions vs. Ground truths.

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Ground Truth")
plt.ylabel("Predictions")
plt.title("Scratch Decision Tree: Predictions vs Ground Truth")
plt.show()

## Fit a decision tree regressor model on the training set using scikit learn. Also, perform the cross validation and find the best values of the two hyperparameters in the same way as explained in case of the from scratch implementation.

In [ ]:
sk_tree = DecisionTreeRegressor(random_state=42)

param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [5, 10, 20],
    "min_samples_leaf": [5],
}

grid_search = GridSearchCV(sk_tree, param_grid, cv=5, scoring="neg_mean_squared_error")
grid_search.fit(X_train, y_train)

print("Best sklearn params:", grid_search.best_params_)

best_sklearn = grid_search.best_estimator_




## Evaluate the performance of the decision tree regressor from scikit-learn on the test set using the best values of the hyperparameters you obtained in step 6. Report the MSE obtained on the test set and show the scatter plot of predictions vs. Ground truths.

In [ ]:
y_pred_test_sklearn = best_sklearn.predict(X_test)
mse_test_sklearn = mean_squared_error(y_test, y_pred_test_sklearn)
print("Sklearn Tree Test MSE:", mse_test_sklearn)

plt.scatter(y_test, y_pred_test_sklearn, alpha=0.5, color="orange")
plt.xlabel("Ground Truth")
plt.ylabel("Predictions")
plt.title("Sklearn Decision Tree: Predictions vs Ground Truth")
plt.show()



NameError: name 'best_sklearn' is not defined

## Visualize the decision tree regressor learned by scikit-learn using scikit-learn's plot-tree: https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html


In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(best_sklearn, feature_names=X.columns, filled=True, fontsize=8)
plt.show()

## Compare the best values of hyperparameters and prediction performance of the from scratch implementation and the scikit-learn implementation of decision tree regressor.

In [ ]:
print("\n=== Comparison ===")
print("Scratch Decision Tree:")
print("  Best Params:", best_params)
print("  Test MSE:", mse_test_scratch)

print("Sklearn Decision Tree:")
print("  Best Params:", grid_search.best_params_)
print("  Test MSE:", mse_test_sklearn)